# Processing Silver layer for account_journal model

In [0]:
%sql

DESCRIBE bakery_pos_develop.bronze.account_journal

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_account_journal AS 
SELECT
    id AS journal_id,
    name AS journal_name,
    type,
    code,
    CAST(create_date AS TIMESTAMP) AS created_at,
    CAST(write_date AS TIMESTAMP) AS updated_at,
    processing_timestamp AS bronze_ingested_at
FROM bakery_pos_develop.bronze.account_journal
QUALIFY ROW_NUMBER() OVER(
    PARTITION BY id
    ORDER BY write_date DESC,
    processing_timestamp DESC
) =1

In [0]:
%sql

CREATE TABLE IF NOT EXISTS bakery_pos_develop.silver.account_journal (
    journal_id INT,
    journal_name STRING,
    type STRING,
    code STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    bronze_ingested_at TIMESTAMP,
    silver_processed_at TIMESTAMP,
    is_updated BOOLEAN,
    is_deleted BOOLEAN
) USING DELTA;

In [0]:
%sql
MERGE INTO bakery_pos_develop.silver.account_journal AS target
USING (
  SELECT *, current_timestamp() AS silver_processed_at
  FROM stg_account_journal
) AS source
ON target.journal_id = source.journal_id
WHEN MATCHED AND source.updated_at > target.updated_at THEN
  UPDATE SET
    journal_name = source.journal_name,
    type = source.type,
    code = source.code,
    created_at = source.created_at,
    updated_at = source.updated_at,
    is_updated = True,
    is_deleted = False,
    bronze_ingested_at = source.bronze_ingested_at,
    silver_processed_at = source.silver_processed_at
WHEN NOT MATCHED THEN
  INSERT (
    journal_id,
    journal_name,
    type,
    code,
    created_at,
    updated_at,
    bronze_ingested_at,
    silver_processed_at,
    is_updated,
    is_deleted
  )
  VALUES (
    source.journal_id,
    source.journal_name,
    source.type,
    source.code,
    source.created_at,
    source.updated_at,
    source.bronze_ingested_at,
    source.silver_processed_at,
    False,
    False
  );